In [1]:
pipeline_names = None  # All pipelines
# pipeline_names = '["pl_ingestion", "pl_transform"]'

In [2]:
import base64
import json
from datetime import datetime, timezone
import requests
from uuid import uuid4

In [3]:
# Always ignored, even if included in pipeline_names
ignore_pipelines = [
    "pl_patch_pipelines"
]

# Check if pipeline names were provided
if isinstance(pipeline_names, str):
    pipeline_names = json.loads(pipeline_names)

if pipeline_names is not None and (
    not isinstance(pipeline_names, list)
    or not all(isinstance(name, str) for name in pipeline_names)
):
    raise ValueError("pipeline_names must be None or a list of pipeline names.")

# Get workspace context
workspace_id = notebookutils.runtime.context["currentWorkspaceId"]

# Get token for current identity
token = notebookutils.credentials.getToken("pbi")

In [ ]:
# Read identity claims from the runtime token for logging only.
payload = token.split(".")[1]
claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
identity_id = claims["oid"]
identity_name = claims.get("name") or claims.get("upn") or identity_id

print(f"Current identity: {identity_name} ({identity_id})")

In [ ]:
# Set session
session = requests.Session()
session.headers["Authorization"] = f"Bearer {token}"

In [ ]:
# Base workspace endpoint
base_url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# List for pipelines found in current workspace
pipelines = []

# Item type to filter only pipelines from other workspace items
params = {"type": "DataPipeline"}

In [ ]:
while True:
    # Get a page of pipelines from the workspace's items endpoint.
    response = session.get(f"{base_url}/items", params=params, timeout=60)

    # Stop with an error if the API request failed.
    response.raise_for_status()

    # Convert the JSON response into a Python dictionary.
    page = response.json()

    # Add the pipelines from this page to our list.
    pipelines.extend(page["value"])

    # Check whether the API has another page of results.
    continuation = page.get("continuationToken")

    # If there are no more pages, exit the loop.
    if not continuation:
        break

    # Include the token in the next request to get the next page.
    params["continuationToken"] = continuation

In [ ]:
# Validate names before making any changes.
if pipeline_names is not None:
    missing = set(pipeline_names) - {p["displayName"] for p in pipelines}
    if missing:
        raise ValueError(f"Pipeline names not found: {sorted(missing)}")


In [ ]:
# Go through the pipelines one at a time.
for pipeline in pipelines:
    # Get the pipeline's name.
    name = pipeline["displayName"]

    # Skip pipelines that are in the ignore list.
    if name in ignore_pipelines:
        print(f"ignored {name}")
        continue

    # If specific names were provided, skip pipelines outside that list.
    if pipeline_names is not None and name not in pipeline_names:
        continue

    # Build the API address for this pipeline.
    url = f"{base_url}/dataPipelines/{pipeline['id']}"

    # Get the pipeline's current details.
    response = session.get(url, timeout=60)
    response.raise_for_status()
    details = response.json()

    # Send the existing name back unchanged.
    # This also leaves the description untouched.
    response = session.patch(
        url,
        json={"displayName": details["displayName"]},
        timeout=60,
    )

    # Stop with an error if the PATCH request failed.
    response.raise_for_status()

    # Print when the PATCH request finished successfully.
    completed_at = datetime.now(timezone.utc).isoformat()
    print(
        f"patched {name} - by {identity_name} "
        f"- PATCH completed at {completed_at}"
    )